In [42]:
import numpy as np
import scipy.io
from scipy.signal import butter, lfilter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# EEGNet-specific imports
from EEGModels import EEGNet
from tensorflow.keras import utils as np_utils
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras import backend as K

import tensorflow.keras as keras
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.constraints import max_norm
from tensorflow.keras import backend as K
import tensorflow.keras.models as models
import tensorflow.compat.v1 as tf
import numpy as np
from tslearn.metrics import soft_dtw
import os
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Activation, Permute, Dropout, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, AveragePooling2D
from tensorflow.keras.layers import SeparableConv2D, DepthwiseConv2D
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import SpatialDropout2D
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.layers import Input, Flatten
from tensorflow.keras.constraints import max_norm
from tensorflow.keras import backend as K
from scipy.signal import butter, filtfilt
from sklearn.model_selection import KFold
from scipy import io, signal
import math


In [4]:
fs = 250
chans=64
kernels=1
n_subjects=35
n_start=500
n_end=750
samples=n_end-n_start
n_freqs=10
n_trials=6

data = {}
for i in range(1, 36):
    file_path = f'D://2024//7-9//dissertation//code//S{i}.mat'
    data[f'S{i}'] = scipy.io.loadmat(file_path)

In [5]:
selected_subjects_keys = [f"S{i}" for i in range(1, n_subjects+1)]

# 初始化一个列表来存储选出的数据
selected_data = []

# 遍历选中的受试者键
for key in selected_subjects_keys:
    # 对每一个受试者数据，选择 9 个通道、2~3 秒 250 个时间点、前 4 个频率、所有实验次数
    subject_data = data[key]['data'][:chans, n_start:n_end, :n_freqs, :]
    selected_data.append(subject_data)

# 将列表转换为 NumPy 数组
selected_data = np.array(selected_data)
X0 = selected_data.transpose(0, 3, 4, 1, 2)
X=X0.reshape(n_subjects*n_freqs*n_trials, chans, samples) # (35 * 40 * 6, 9, 500)

y = []

for i in range(n_subjects):
    for j in range(n_freqs):
        for k in range(n_trials):
            y.append(j)
            
y = np_utils.to_categorical(y)

In [6]:
def bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs  # Nyquist Frequency
    low = lowcut / nyq
    high = highcut /nyq
    b, a = butter(order, [low, high], btype='band')
    y = lfilter(b, a, data)
    return y

In [14]:
X_train, X_test, Y_train,Y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

X1_train = np.zeros_like(X_train)
X2_train = np.zeros_like(X_train)
X3_train = np.zeros_like(X_train)
for i in range(X_train.shape[0]):
    for j in range(X_train.shape[1]):
        X1_train[i, j, :] = bandpass_filter(X_train[i, j, :], 7, 90, fs, order=5)
        X2_train[i, j, :] = bandpass_filter(X_train[i, j, :], 8, 90, fs, order=5)
        X3_train[i, j, :] = bandpass_filter(X_train[i, j, :], 9, 90, fs, order=5)

X1_test = np.zeros_like(X_test)
X2_test = np.zeros_like(X_test)
X3_test = np.zeros_like(X_test)
for i in range(X_test.shape[0]):
    for j in range(X_test.shape[1]):
        X1_test[i, j, :] = bandpass_filter(X_test[i, j, :], 7, 90, fs, order=5)
        X2_test[i, j, :] = bandpass_filter(X_test[i, j, :], 8, 90, fs, order=5)
        X3_test[i, j, :] = bandpass_filter(X_test[i, j, :], 9, 90, fs, order=5)

In [23]:
############################# EEGNet portion ##################################

# convert data to NHWC (trials, channels, samples, kernels) format. Data 
# contains 60 channels and 151 time-points. Set the number of kernels to 1.
X1_train = X1_train.reshape(X1_train.shape[0], chans, samples, kernels)
X2_train = X1_train.reshape(X2_train.shape[0], chans, samples, kernels)
X3_train = X3_train.reshape(X3_train.shape[0], chans, samples, kernels)
X1_test  = X1_test.reshape(X1_test.shape[0], chans, samples, kernels)
X2_test  = X2_test.reshape(X2_test.shape[0], chans, samples, kernels)
X3_test  = X3_test.reshape(X3_test.shape[0], chans, samples, kernels)

# configure the EEGNet-8,2,16 model with kernel length of 32 samples (other 
# model configurations may do better, but this is a good starting point)
model1 = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, 
               dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')
model2 = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, 
               dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')
model3 = EEGNet(nb_classes = n_freqs, Chans = chans, Samples = samples, 
               dropoutRate = 0.5, kernLength = 125, F1 = 8, D = 2, F2 = 16, 
               dropoutType = 'Dropout')

# compile the model and set the optimizers
model1.compile(loss='categorical_crossentropy', optimizer='adam', 
              metrics = ['accuracy'])
model2.compile(loss='categorical_crossentropy', optimizer='adam', 
              metrics = ['accuracy'])
model3.compile(loss='categorical_crossentropy', optimizer='adam', 
              metrics = ['accuracy'])

# count number of parameters in the model
numParams1    = model1.count_params()
numParams2    = model2.count_params()   
numParams3    = model3.count_params()   

# set a valid path for your system to record model checkpoints
checkpointer = ModelCheckpoint(filepath='/tmp/checkpoint.keras', verbose=1,
                               save_best_only=True)

# fit the models
fittedModel1 = model1.fit(X1_train, Y_train, batch_size = 16, epochs = 50,verbose = 2, validation_split = 0.2)
probs1       = model1.predict(X1_test)
preds1       = probs1.argmax(axis = -1)  
acc1         = np.mean(preds1 == Y_test.argmax(axis=-1))
print("Classification accuracy: %f " % (acc1))

fittedModel2 = model2.fit(X2_train, Y_train, batch_size = 16, epochs = 50,verbose = 2, validation_split = 0.2)
probs2       = model2.predict(X2_test)
preds2       = probs2.argmax(axis = -1)  
acc2         = np.mean(preds2 == Y_test.argmax(axis=-1))
print("Classification accuracy: %f " % (acc2))

fittedModel3 = model3.fit(X3_train, Y_train, batch_size = 16, epochs = 50,verbose = 2, validation_split = 0.2)
probs3       = model3.predict(X3_test)
preds3       = probs3.argmax(axis = -1)  
acc3         = np.mean(preds3 == Y_test.argmax(axis=-1))
print("Classification accuracy: %f " % (acc3))

Epoch 1/50
84/84 - 7s - 80ms/step - accuracy: 0.1042 - loss: 2.3196 - val_accuracy: 0.1250 - val_loss: 2.3011
Epoch 2/50
84/84 - 5s - 57ms/step - accuracy: 0.1310 - loss: 2.2938 - val_accuracy: 0.1577 - val_loss: 2.2952
Epoch 3/50
84/84 - 5s - 57ms/step - accuracy: 0.1577 - loss: 2.2713 - val_accuracy: 0.2619 - val_loss: 2.2675
Epoch 4/50
84/84 - 5s - 58ms/step - accuracy: 0.2470 - loss: 2.2423 - val_accuracy: 0.3363 - val_loss: 2.2096
Epoch 5/50
84/84 - 5s - 60ms/step - accuracy: 0.2984 - loss: 2.1897 - val_accuracy: 0.4673 - val_loss: 2.1455
Epoch 6/50
84/84 - 5s - 60ms/step - accuracy: 0.3698 - loss: 2.1327 - val_accuracy: 0.4196 - val_loss: 2.0718
Epoch 7/50
84/84 - 5s - 59ms/step - accuracy: 0.3914 - loss: 2.0711 - val_accuracy: 0.4702 - val_loss: 1.9932
Epoch 8/50
84/84 - 5s - 61ms/step - accuracy: 0.3772 - loss: 2.0036 - val_accuracy: 0.4732 - val_loss: 1.9437
Epoch 9/50
84/84 - 5s - 65ms/step - accuracy: 0.3810 - loss: 1.9827 - val_accuracy: 0.5327 - val_loss: 1.9101
Epoch 10/5

In [39]:
'''for layer in model1.layers:
    layer.name = layer.name + "a"

for layer in model2.layers:
    layer.name = layer.name + "b"

for layer in model3.layers:
    layer.name = layer.name + "c"'''

(1680, 64, 250, 1)

In [41]:
model1.trainable = False
model2.trainable = False
model3.trainable = False

features = layers.concatenate([model1.layers[-2].output, model2.layers[-2].output, model3.layers[-2].output])
features = layers.Dense(10, activation="softmax")(features)

fusion_model = models.Model([model1.input, model2.input, model3.input], features)

fusion_model.compile(loss='categorical_crossentropy', optimizer = "Adam", metrics = ["accuracy"])

fusion_model.summary()

fusion_model.fit([X1_train, X2_train, X3_train], Y_train, batch_size=16, epochs=50, shuffle=True, verbose=1, 
                 validation_data=([X1_test, X2_test, X3_test], Y_test))
score_fusion = fusion_model.evaluate([X1_test, X2_test, X3_test], Y_test, verbose=0)
print(f"FB-EEGNET acc: {score_fusion[1]}", flush=True)

Model: "functional_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_12a (InputLayer)  │ (None, 64, 250, 1)        │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_13b (InputLayer)  │ (None, 64, 250, 1)        │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_14c (InputLayer)  │ (None, 64, 250, 1)        │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_12a (Conv2D)           │ (None, 64, 250, 8)        │           1,000 │ input_layer_12a[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_13b (Conv2D)           │ (None, 64, 250, 8)        │           1,000 │ input_layer_13b[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_14c (Conv2D)           │ (None, 64, 250, 8)        │           1,000 │ input_layer_14c[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_36a       │ (None, 64, 250, 8)        │              32 │ conv2d_12a[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_39b       │ (None, 64, 250, 8)        │              32 │ conv2d_13b[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_42c       │ (None, 64, 250, 8)        │              32 │ conv2d_14c[0][0]           │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ depthwise_conv2d_12a          │ (None, 1, 250, 16)        │           1,024 │ batch_normalization_36a[0… │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ depthwise_conv2d_13b          │ (None, 1, 250, 16)        │           1,024 │ batch_normalization_39b[0… │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ depthwise_conv2d_14c          │ (None, 1, 250, 16)        │           1,024 │ batch_normalization_42c[0… │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_37a       │ (None, 1, 250, 16)        │              64 │ depthwise_conv2d_12a[0][0] │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_40b       │ (None, 1, 250, 16)        │              6

 Total params: 11,788 (46.05 KB)

 Trainable params: 310 (1.21 KB)

 Non-trainable params: 11,478 (44.84 KB)

Epoch 1/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 7s 53ms/step - accuracy: 0.1858 - loss: 3.3828 - val_accuracy: 0.3976 - val_loss: 1.8431
Epoch 2/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.3641 - loss: 1.8383 - val_accuracy: 0.5714 - val_loss: 1.2433
Epoch 3/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 51ms/step - accuracy: 0.5130 - loss: 1.2950 - val_accuracy: 0.6524 - val_loss: 1.0888
Epoch 4/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.5906 - loss: 1.1086 - val_accuracy: 0.6738 - val_loss: 1.0222
Epoch 5/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.6278 - loss: 1.0023 - val_accuracy: 0.6929 - val_loss: 0.9831
Epoch 6/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.6428 - loss: 0.9400 - val_accuracy: 0.7071 - val_loss: 0.9536
Epoch 7/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.6883 - loss: 0.8826 - val_accuracy: 0.7119 - val_loss: 0.9379
Epoch 8/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.6671 - loss: 0.8910 - val_accu